<a href="https://colab.research.google.com/github/adhikaryramen87/MachineLearning_Works/blob/main/Building_Reproducible_Training_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Fit the reproducibility setup

Import all the required libraries & used the seed values

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf

SEED = 42

# 1. Python randomness
random.seed(SEED)

# 2. Numpy randomness
np.random.seed(SEED)

# 3. TensorFlow randomness
tf.random.set_seed(SEED)

# 4. Force deterministic ops (important for GPU)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

print("Seeds set for reproducibility!")

Seeds set for reproducibility!


Load and Preprocess the MNIST data

In [ ]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Load data
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalize
X_train = X_train / 255.0
X_test = X_test / 255.0

# Reshape for ANN -> Flattening
X_train = X_train.reshape(-1, 28*28)
X_test = X_test.reshape(-1, 28*28)

# One-hot encoding
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Config-Driven Model -> Hyperparameter Tuning

In [ ]:
config = {
    "learning_rate": 0.001,
    "epochs": 5,
    "batch_size": 32,
    "n_neurons": 128,
    "dropout_rate": 0.2
}

Build the Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

def build_model(config):
    model = Sequential([
        Dense(config["n_neurons"], activation='relu', input_shape=(784,)),
        Dropout(config["dropout_rate"]),
        Dense(64, activation='relu'),
        Dense(10, activation='softmax')
    ])

    optimizer = Adam(learning_rate=config["learning_rate"])

    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

You need to introduce the Pipeline Method

```python
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', model)
])
```

Training Pipeline

In [ ]:
def train_pipeline(config):

    # Reset seeds AGAIN (important for multiple runs)
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    model = build_model(config)

    history = model.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=config["epochs"],
        batch_size=config["batch_size"],
        verbose=0
    )

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    return test_acc

Same Results => Reproducible Pipeline

Different Results => Something which is still uncontrolled

In [ ]:
results = []

for i in range(3):
    acc = train_pipeline(config)
    results.append(acc)
    print(f"Run {i+1}: Accuracy = {acc:.4f}")

print("\nAll Results:", results)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Run 1: Accuracy = 0.9766
Run 2: Accuracy = 0.9766
Run 3: Accuracy = 0.9766

All Results: [0.9765999913215637, 0.9765999913215637, 0.9765999913215637]


Save the artifacts

In [ ]:
import json

# Save config
with open("config.json", "w") as f:
    json.dump(config, f)

# Save model
model = build_model(config)
model.save("mnist_model.h5")